<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_01_05_hpc_bigdata_postgresql_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Working with PostgreSQL in R




[PostgreSQL](https://www.postgresql.org/) is a powerful, open-source, object-relational database management system (ORDBMS) that has been actively developed for over 35 years. It is known for its robustness, extensibility, SQL standards compliance, and ability to handle large-scale data workloads — making it a favourite choice for analysts, engineers, and data scientists alike.

Unlike lightweight databases such as SQLite, PostgreSQL is a full-featured server-based database system designed for multi-user concurrency, complex queries, and terabyte-scale datasets.



### Why Use PostgreSQL with R?

R is exceptional for statistical computing and data analysis. PostgreSQL excels at storing, querying, and managing large structured datasets. Together they form a powerful combination:

- **Offload heavy queries** to the database engine, pulling only summarised results into R
- **Persist datasets** that are too large to hold in R's memory
- **Share data** across teams and tools using a central database server
- **Leverage SQL** for filtering, joining, grouping, and aggregating before analysis



### What This Tutorial Covers

| Section | Topic |
|---------|-------|
| 1 | Installing PostgreSQL on Ubuntu and Windows |
| 2 | Creating databases, users, and tables |
| 3 | Connecting from R using `DBI` + `RPostgres` |
| 4 | Basic CRUD operations |
| 5 | Importing CSV and Parquet datasets |
| 6 | Advanced SQL: JOINs, window functions, CTEs |
| 7 | Using `dbplyr` for dplyr-style queries |
| 8 | Performance tuning with indexes and EXPLAIN |
| 9 | Security best practices |
| 10 | Disconnecting and cleanup |



### Datasets Used

This tutorial uses two freely available NYC Taxi datasets:

| File | Description | Size |
|------|-------------|------|
| `yellow_tripdata_2023-01.parquet` | NYC Yellow Taxi trip records — January 2023 | ~50 MB |
| `taxi_zone_lookup.csv` | Mapping of location IDs to borough and zone names | ~12 KB |

Both can be downloaded from the [NYC TLC Trip Record Data page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).




## Installing PostgreSQL

### Ubuntu / Debian Linux

Open a terminal and run the following commands in sequence.

**Step 1 — Update your package index:**

```bash
sudo apt update
```

**Step 2 — Install PostgreSQL and contrib extensions:**

```bash
sudo apt install postgresql postgresql-contrib -y
```

**Step 3 — Start the service and enable it on boot:**

```bash
sudo systemctl start postgresql
sudo systemctl enable postgresql
```

**Step 4 — Set a password for the default `postgres` superuser:**

```bash
sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'your_secure_password';"
```

**Step 5 — Verify the installation:**

```bash
sudo -u postgres psql -c "\conninfo"
```

Expected output:

```
You are connected to database "postgres" as user "postgres"
via socket in "/var/run/postgresql" at port "5432".
```

> **Tip: Installing a Specific Version**
>
To install the latest stable release (e.g. PostgreSQL 17) from the official PGDG repository:

```bash
sudo sh -c 'echo "deb https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" \
  > /etc/apt/sources.list.d/pgdg.list'
curl -fsSL https://www.postgresql.org/media/keys/ACCC4CF8.asc | sudo gpg --dearmor \
  -o /etc/apt/trusted.gpg.d/postgresql.gpg
sudo apt update
sudo apt install postgresql-17 -y
```

### Windows

1. Download the installer from [postgresql.org/download/windows](https://www.postgresql.org/download/windows/) — choose the latest version.
2. Run the installer **as Administrator**.
3. During setup, select:
   - ✅ PostgreSQL Server
   - ✅ pgAdmin 4 (GUI management tool)
   - ✅ Command Line Tools
4. Set a password for the `postgres` superuser — **save this carefully**.
5. Keep the default port **5432**.
6. After installation, add the bin directory to your system PATH:
   - `C:\Program Files\PostgreSQL\17\bin`
   - Go to **System Properties → Environment Variables → Path → Edit**.
7. Start the service from **Services** (`services.msc`) — find `postgresql-x64-17` and start it.

**Verify from Command Prompt:**

```cmd
psql -U postgres -h localhost
```

Type `\conninfo` at the prompt to confirm.

---



## Creating a Database and Schema

### Via the psql Command Line

Connect to the PostgreSQL shell:

- **Ubuntu:** `sudo -u postgres psql`
- **Windows:** `psql -U postgres -h localhost`

Then run:

```sql
-- Create the database
CREATE DATABASE taxi_db;

-- Connect to it
\c taxi_db

-- Create a dedicated user (best practice — avoid using postgres superuser)
CREATE USER r_user WITH PASSWORD 'r_secure_pass';

-- Grant privileges on the database
GRANT ALL PRIVILEGES ON DATABASE taxi_db TO r_user;

-- Grant schema-level privileges (PostgreSQL 15+)
GRANT ALL ON SCHEMA public TO r_user;

-- Exit
\q
```



### Via pgAdmin (GUI)

pgAdmin is the official graphical tool for PostgreSQL, installed alongside the server.

1. Open pgAdmin and connect to your local server.
2. Right-click **Databases → Create → Database**, name it `taxi_db`.
3. Under **Login/Group Roles**, create a new role `r_user` with a password and login permission.
4. Use the **Query Tool** (lightning bolt icon) to run SQL statements.

> **Note: pgAdmin is Recommended for Beginners**
>
pgAdmin provides point-and-click database management, a visual query editor with syntax highlighting, and tools to inspect table structure and data — much easier than working entirely in the terminal.



## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab's Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Drive if your data files or R library live there. Adjust paths in the data-folder cell to match your Drive layout.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Required R Packages

In [ ]:
%%R
#| label: packages

packages <- c(
  "DBI",          # Unified database interface
  "RPostgres",    # PostgreSQL driver for R
  "tidyverse",    # Data wrangling (includes dplyr, readr, ggplot2, etc.)
  "dbplyr",       # Translate dplyr to SQL (installed with tidyverse)
  "arrow",        # Read Parquet files efficiently
  "data.table",   # High-performance data manipulation
  "glue",         # Safe SQL string interpolation
  "keyring"       # Securely store database passwords
)


### Install Missing Packages

```r
#| label: install-packages

new_packages <- packages[!(packages %in% installed.packages()[, "Package"])]
if (length(new_packages) > 0) {
  install.packages(new_packages)
}
```

### Load Libraries


In [ ]:
%%R
#| label: load-libraries

library(DBI)
library(RPostgres)
library(tidyverse)
library(dbplyr)
library(arrow)
library(data.table)
library(glue)


### Verify Package Versions


In [ ]:
%%R
#| label: verify-packages

pkg_versions <- sapply(packages, function(p) {
  if (p %in% installed.packages()[, "Package"]) {
    as.character(packageVersion(p))
  } else {
    "NOT INSTALLED"
  }
})

data.frame(
  Package = names(pkg_versions),
  Version = pkg_versions,
  row.names = NULL
)


## Connecting to PostgreSQL from R

### About DBI and RPostgres

R uses a two-layer architecture for database connections:

- **`DBI`** — Defines a standard interface (functions like `dbConnect()`, `dbGetQuery()`, `dbWriteTable()`) that works with any supported database.
- **`RPostgres`** — Provides the PostgreSQL-specific driver. Built in C++ via Rcpp for high performance.

This means code using `DBI` functions is largely portable — you can switch databases (PostgreSQL → SQLite → MySQL) by changing only the driver.

### Basic Connection


In [ ]:
%%R
#| label: connect-basic

con <- dbConnect(
  drv      = RPostgres::Postgres(),
  dbname   = "taxi_db",
  host     = "localhost",
  port     = 5432,
  user     = "r_user",
  password = "r_secure_pass"   # See security section for better approaches
)

# Verify connection
dbGetQuery(con, "SELECT version();")


### Secure Connections Using .Renviron

**Never hardcode passwords in scripts.** Instead, store credentials in your `.Renviron` file:

```bash
# In ~/.Renviron (one entry per line, no spaces around =)
PG_HOST=localhost
PG_PORT=5432
PG_DB=taxi_db
PG_USER=r_user
PG_PASSWORD=r_secure_pass
```

Then in R:


In [ ]:
%%R
#| label: connect-secure

con <- dbConnect(
  drv      = RPostgres::Postgres(),
  dbname   = Sys.getenv("PG_DB"),
  host     = Sys.getenv("PG_HOST"),
  port     = as.integer(Sys.getenv("PG_PORT")),
  user     = Sys.getenv("PG_USER"),
  password = Sys.getenv("PG_PASSWORD")
)


> **Tip: Reload .Renviron Without Restarting R**
>
```r
readRenviron("~/.Renviron")
```

### Using `keyring` for OS-Level Secret Storage

The `keyring` package stores secrets in your operating system's secure credential store (macOS Keychain, Windows Credential Manager, Linux Secret Service):


In [ ]:
%%R
#| label: connect-keyring

library(keyring)

# Store the password once (run interactively)
# key_set("postgresql", username = "r_user")

# Then connect using the stored secret
con <- dbConnect(
  drv      = RPostgres::Postgres(),
  dbname   = "taxi_db",
  host     = "localhost",
  port     = 5432,
  user     = "r_user",
  password = key_get("postgresql", username = "r_user")
)


### Check Connection Status


In [ ]:
%%R
#| label: check-connection

# List tables in the connected database
dbListTables(con)

# Check if a specific table exists
dbExistsTable(con, "taxi_zones")

# Get connection info
dbGetInfo(con)


---

## Basic CRUD Operations

CRUD stands for **C**reate, **R**ead, **U**pdate, **D**elete — the four fundamental database operations.

### Create a Table


In [ ]:
%%R
#| label: create-table

dbExecute(con, "
  CREATE TABLE IF NOT EXISTS employees (
    id         SERIAL PRIMARY KEY,
    name       VARCHAR(100) NOT NULL,
    department VARCHAR(50),
    salary     NUMERIC(10, 2),
    hired_on   DATE DEFAULT CURRENT_DATE
  );
")


> **Note: Key SQL Data Types in PostgreSQL**
>

| R Type | PostgreSQL Type | Notes |
|--------|----------------|-------|
| `integer` | `INTEGER` or `SERIAL` | `SERIAL` auto-increments |
| `numeric` | `NUMERIC(p, s)` | Exact decimal; use for money |
| `character` | `VARCHAR(n)` | Variable-length string |
| `logical` | `BOOLEAN` | TRUE / FALSE |
| `Date` | `DATE` | YYYY-MM-DD |
| `POSIXct` | `TIMESTAMP` | Date + time |

### Insert Data

**Single row using SQL:**


In [ ]:
%%R
#| label: insert-single

dbExecute(con, "
  INSERT INTO employees (name, department, salary, hired_on)
  VALUES ('Alice Chen', 'Analytics', 85000.00, '2022-03-15');
")


**Multiple rows at once:**


In [ ]:
%%R
#| label: insert-multiple

dbExecute(con, "
  INSERT INTO employees (name, department, salary, hired_on) VALUES
    ('Bob Smith',    'Engineering', 92000.00, '2021-07-01'),
    ('Carol White',  'Analytics',   78000.00, '2023-01-10'),
    ('David Lee',    'Marketing',   67000.00, '2020-11-22'),
    ('Eve Johnson',  'Engineering', 95000.00, '2019-06-30');
")


**Write an R data frame directly (most practical approach):**


In [ ]:
%%R
#| label: insert-dataframe

new_hires <- data.frame(
  name       = c("Frank Brown", "Grace Kim"),
  department = c("Analytics", "Engineering"),
  salary     = c(81000.00, 88000.00),
  hired_on   = as.Date(c("2024-02-01", "2024-03-15"))
)

dbWriteTable(
  conn      = con,
  name      = "employees",
  value     = new_hires,
  append    = TRUE,   # Add to existing table
  row.names = FALSE
)


### Read Data

**Fetch all rows:**


In [ ]:
%%R
#| label: read-all

employees <- dbGetQuery(con, "SELECT * FROM employees ORDER BY id;")
print(employees)


**Fetch with conditions:**


In [ ]:
%%R
#| label: read-filtered

analytics_team <- dbGetQuery(con, "
  SELECT name, salary
  FROM employees
  WHERE department = 'Analytics'
  ORDER BY salary DESC;
")
print(analytics_team)


**Parameterised queries (safe against SQL injection):**


In [ ]:
%%R
#| label: read-parameterised

dept <- "Engineering"

result <- dbGetQuery(
  conn    = con,
  statement = "SELECT * FROM employees WHERE department = $1;",
  params  = list(dept)
)
print(result)


> **Warning: Always Use Parameterised Queries for User Input**
>
Never build SQL strings using `paste()` with user-supplied values. This opens the door to SQL injection attacks. Use `$1`, `$2`, ... placeholders with the `params` argument instead.

**Fetch a single value:**


In [ ]:
%%R
#| label: read-scalar

avg_salary <- dbGetQuery(con, "SELECT ROUND(AVG(salary), 2) AS avg_salary FROM employees;")
cat("Average salary: $", avg_salary$avg_salary, "\n")


**Stream large results (low memory):**


In [ ]:
%%R
#| label: read-stream

# Use dbSendQuery + dbFetch for chunked reading
query <- dbSendQuery(con, "SELECT * FROM employees;")

chunks <- list()
while (!dbHasCompleted(query)) {
  chunk <- dbFetch(query, n = 3)   # Read 3 rows at a time
  chunks <- c(chunks, list(chunk))
}
dbClearResult(query)

all_data <- bind_rows(chunks)
print(all_data)


### Update Data


In [ ]:
%%R
#| label: update-data

# Give all Analytics employees a 10% raise
rows_affected <- dbExecute(con, "
  UPDATE employees
  SET salary = salary * 1.10
  WHERE department = 'Analytics';
")
cat("Rows updated:", rows_affected, "\n")

# Verify
dbGetQuery(con, "SELECT name, salary FROM employees WHERE department = 'Analytics';")


### Delete Data


In [ ]:
%%R
#| label: delete-data

# Delete employees hired before 2021
dbExecute(con, "
  DELETE FROM employees
  WHERE hired_on < '2021-01-01';
")

# Confirm deletion
dbGetQuery(con, "SELECT COUNT(*) AS remaining FROM employees;")


### Drop a Table


In [ ]:
%%R
#| label: drop-table

# Only drop if it exists (safe to re-run)
dbExecute(con, "DROP TABLE IF EXISTS employees;")


---

## Importing Real-World Datasets

We now import the two NYC taxi datasets into PostgreSQL.

### Setup: File Paths


### Data folder before import

The next cell sets `data_folder` (default `~/data/nyc_taxi/` from the Quarto source). On **Colab**, change it to a Drive path such as `/content/drive/MyDrive/YourFolder/Data/` after mounting Drive.


In [ ]:
%%R
#| label: file-paths

data_folder <- "~/data/nyc_taxi/"   # Adjust to your path

taxi_zones_file    <- file.path(data_folder, "taxi_zone_lookup.csv")
yellow_trips_file  <- file.path(data_folder, "yellow_tripdata_2023-01.parquet")


### Import taxi_zone_lookup.csv

This is a small reference table (~265 rows). Read with `readr` and write directly.


In [ ]:
%%R
#| label: import-zones

# Read CSV into R
zones <- read_csv(taxi_zones_file, col_types = cols(
  LocationID   = col_integer(),
  Borough      = col_character(),
  Zone         = col_character(),
  service_zone = col_character()
))

# Rename to snake_case for PostgreSQL
names(zones) <- tolower(names(zones))
names(zones)[names(zones) == "locationid"] <- "location_id"

# Create the table (explicit schema is better than letting DBI infer)
dbExecute(con, "DROP TABLE IF EXISTS taxi_zones;")
dbExecute(con, "
  CREATE TABLE taxi_zones (
    location_id  INTEGER PRIMARY KEY,
    borough      VARCHAR(50),
    zone         VARCHAR(100),
    service_zone VARCHAR(50)
  );
")

# Insert data
dbWriteTable(con, "taxi_zones", zones, append = TRUE, row.names = FALSE)

# Verify
dbGetQuery(con, "SELECT COUNT(*) AS n_zones FROM taxi_zones;")
dbGetQuery(con, "SELECT * FROM taxi_zones LIMIT 5;")


### Import yellow_tripdata_2023-01.parquet (Large File)

Parquet files can contain millions of rows. We use `arrow` to read in batches and insert chunk-by-chunk to stay within memory limits.


In [ ]:
%%R
#| label: create-trips-table

# Drop and recreate with explicit types
dbExecute(con, "DROP TABLE IF EXISTS yellow_trips;")

dbExecute(con, "
  CREATE TABLE yellow_trips (
    vendorid                INTEGER,
    tpep_pickup_datetime    TIMESTAMP,
    tpep_dropoff_datetime   TIMESTAMP,
    passenger_count         INTEGER,
    trip_distance           NUMERIC(8, 2),
    ratecodeid              INTEGER,
    store_and_fwd_flag      VARCHAR(1),
    pulocationid            INTEGER,
    dolocationid            INTEGER,
    payment_type            INTEGER,
    fare_amount             NUMERIC(10, 2),
    extra                   NUMERIC(8, 2),
    mta_tax                 NUMERIC(8, 2),
    tip_amount              NUMERIC(10, 2),
    tolls_amount            NUMERIC(10, 2),
    improvement_surcharge   NUMERIC(8, 2),
    total_amount            NUMERIC(10, 2),
    congestion_surcharge    NUMERIC(8, 2),
    airport_fee             NUMERIC(8, 2)
  );
")


In [ ]:
%%R
#| label: import-trips-chunked

# Open Parquet as an Arrow Dataset (does NOT load into memory yet)
dataset <- arrow::open_dataset(yellow_trips_file)

cat("Total rows in Parquet file:", nrow(dataset), "\n")
cat("Columns:", paste(names(dataset), collapse = ", "), "\n")

# Define chunk size (rows per batch; adjust based on available RAM)
chunk_size <- 100000

# Collect full data into R — for large files use batched approach below
trips_df <- dataset |>
  collect() |>
  rename_with(tolower)   # lowercase all column names

# Insert in chunks to PostgreSQL
n_rows  <- nrow(trips_df)
n_chunks <- ceiling(n_rows / chunk_size)

cat("Inserting", n_rows, "rows in", n_chunks, "chunks...\n")

for (i in seq_len(n_chunks)) {
  start_row <- (i - 1) * chunk_size + 1
  end_row   <- min(i * chunk_size, n_rows)
  chunk     <- trips_df[start_row:end_row, ]

  dbWriteTable(con, "yellow_trips", chunk, append = TRUE, row.names = FALSE)
  cat("  Chunk", i, "/", n_chunks, "inserted (rows", start_row, "-", end_row, ")\n")
}

cat("Import complete.\n")


> **Tip: Alternative: Very Large Files (> 1 GB)**
>
For files too large to fit in memory, use Arrow's RecordBatch streaming reader:

```r
reader <- arrow::ParquetFileReader$create(yellow_trips_file)
n_groups <- reader$num_row_groups

for (i in seq_len(n_groups)) {
  batch <- reader$ReadRowGroup(i - 1)   # 0-indexed
  chunk <- as.data.frame(batch)
  names(chunk) <- tolower(names(chunk))
  dbWriteTable(con, "yellow_trips", chunk, append = TRUE, row.names = FALSE)
  cat("Row group", i, "of", n_groups, "inserted\n")
}
```

### Verify Imported Data


In [ ]:
%%R
#| label: verify-import

# Row counts
dbGetQuery(con, "SELECT COUNT(*) AS trip_count FROM yellow_trips;")
dbGetQuery(con, "SELECT COUNT(*) AS zone_count FROM taxi_zones;")

# Preview trips
dbGetQuery(con, "SELECT * FROM yellow_trips LIMIT 5;")

# Check date range
dbGetQuery(con, "
  SELECT
    MIN(tpep_pickup_datetime) AS earliest,
    MAX(tpep_pickup_datetime) AS latest
  FROM yellow_trips;
")


---

## Advanced SQL Queries from R

### JOINs — Trips by Borough


In [ ]:
%%R
#| label: query-by-borough

trips_by_borough <- dbGetQuery(con, "
  SELECT
    z.borough,
    COUNT(*)                         AS trip_count,
    ROUND(AVG(t.fare_amount), 2)     AS avg_fare,
    ROUND(AVG(t.trip_distance), 2)   AS avg_distance_miles
  FROM yellow_trips t
  JOIN taxi_zones z
    ON t.pulocationid = z.location_id
  GROUP BY z.borough
  ORDER BY trip_count DESC;
")

print(trips_by_borough)


### Time-Series — Trips per Hour of Day


In [ ]:
%%R
#| label: query-hourly

hourly_demand <- dbGetQuery(con, "
  SELECT
    EXTRACT(HOUR FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*)                                AS trip_count
  FROM yellow_trips
  GROUP BY pickup_hour
  ORDER BY pickup_hour;
")

# Plot
ggplot(hourly_demand, aes(x = pickup_hour, y = trip_count)) +
  geom_col(fill = "#2C7BB6", colour = "white") +
  scale_x_continuous(breaks = 0:23) +
  scale_y_continuous(labels = scales::comma) +
  labs(
    title    = "NYC Yellow Taxi Demand by Hour of Day (January 2023)",
    subtitle = "Total trip pickups per hour",
    x        = "Hour of Day (0 = midnight)",
    y        = "Number of Trips"
  ) +
  theme_minimal(base_size = 13)


### Window Functions — Top Zones by Average Tip

Window functions let you compute rankings, running totals, and moving averages without collapsing rows:


In [ ]:
%%R
#| label: query-window-functions

top_tip_zones <- dbGetQuery(con, "
  WITH zone_tips AS (
    SELECT
      z.zone,
      z.borough,
      ROUND(AVG(t.tip_amount), 2)     AS avg_tip,
      ROUND(AVG(t.fare_amount), 2)    AS avg_fare,
      COUNT(*)                        AS trip_count
    FROM yellow_trips t
    JOIN taxi_zones z ON t.dolocationid = z.location_id
    WHERE t.payment_type = 1       -- Credit card (tips are recorded)
      AND t.trip_distance > 0
    GROUP BY z.zone, z.borough
    HAVING COUNT(*) > 100          -- Minimum 100 trips for reliability
  )
  SELECT
    zone,
    borough,
    avg_tip,
    avg_fare,
    trip_count,
    RANK() OVER (ORDER BY avg_tip DESC) AS rank_by_tip
  FROM zone_tips
  ORDER BY rank_by_tip
  LIMIT 15;
")

print(top_tip_zones)


### Common Table Expressions (CTEs) — Peak vs Off-Peak Comparison


In [ ]:
%%R
#| label: query-cte-peak

peak_comparison <- dbGetQuery(con, "
  WITH trip_periods AS (
    SELECT
      fare_amount,
      tip_amount,
      trip_distance,
      CASE
        WHEN EXTRACT(HOUR FROM tpep_pickup_datetime) BETWEEN 7  AND 9  THEN 'Morning Peak'
        WHEN EXTRACT(HOUR FROM tpep_pickup_datetime) BETWEEN 16 AND 19 THEN 'Evening Peak'
        ELSE 'Off-Peak'
      END AS period
    FROM yellow_trips
    WHERE fare_amount > 0 AND trip_distance > 0
  )
  SELECT
    period,
    COUNT(*)                          AS trips,
    ROUND(AVG(fare_amount), 2)        AS avg_fare,
    ROUND(AVG(tip_amount), 2)         AS avg_tip,
    ROUND(AVG(trip_distance), 2)      AS avg_distance
  FROM trip_periods
  GROUP BY period
  ORDER BY avg_fare DESC;
")

print(peak_comparison)


### Payment Type Analysis


In [ ]:
%%R
#| label: query-payment

payment_labels <- data.frame(
  payment_type = 1:6,
  payment_name = c("Credit Card", "Cash", "No Charge",
                   "Dispute", "Unknown", "Voided Trip")
)

payment_summary <- dbGetQuery(con, "
  SELECT
    payment_type,
    COUNT(*)                      AS trip_count,
    ROUND(SUM(fare_amount), 2)    AS total_fare,
    ROUND(AVG(tip_amount), 2)     AS avg_tip
  FROM yellow_trips
  GROUP BY payment_type
  ORDER BY trip_count DESC;
")

payment_summary <- payment_summary |>
  left_join(payment_labels, by = "payment_type") |>
  select(payment_name, everything(), -payment_type)

print(payment_summary)


---

## Using dbplyr for dplyr-Style Database Queries

`dbplyr` translates `dplyr` verbs into SQL automatically. This is ideal for analysts more comfortable with tidyverse syntax.

### Create Lazy Table References


In [ ]:
%%R
#| label: dbplyr-setup

# These don't load data — they just reference the tables
trips_tbl <- tbl(con, "yellow_trips")
zones_tbl <- tbl(con, "taxi_zones")

# Show dimensions
trips_tbl |> count()    # Runs SELECT COUNT(*) ...


### Filter, Select, and Summarise


In [ ]:
%%R
#| label: dbplyr-wrangle

avg_fare_by_borough <- trips_tbl |>
  left_join(zones_tbl, by = c("pulocationid" = "location_id")) |>
  filter(trip_distance > 1, fare_amount > 0) |>
  group_by(borough) |>
  summarise(
    trips        = n(),
    avg_fare     = mean(fare_amount, na.rm = TRUE),
    avg_distance = mean(trip_distance, na.rm = TRUE),
    avg_tip      = mean(tip_amount, na.rm = TRUE)
  ) |>
  arrange(desc(trips)) |>
  collect()   # <-- Executes query and pulls result into R

print(avg_fare_by_borough)


### Inspect the Generated SQL


In [ ]:
%%R
#| label: dbplyr-show-sql

query <- trips_tbl |>
  filter(trip_distance > 5) |>
  group_by(payment_type) |>
  summarise(avg_fare = mean(fare_amount, na.rm = TRUE))

# Print the SQL without executing
show_query(query)


### Time-Based Grouping with dbplyr


In [ ]:
%%R
#| label: dbplyr-time

daily_trips <- trips_tbl |>
  mutate(trip_date = sql("tpep_pickup_datetime::date")) |>
  group_by(trip_date) |>
  summarise(
    trips    = n(),
    revenue  = sum(total_amount, na.rm = TRUE)
  ) |>
  arrange(trip_date) |>
  collect()

# Plot
ggplot(daily_trips, aes(x = as.Date(trip_date), y = trips)) +
  geom_line(colour = "#E63946", linewidth = 0.8) +
  geom_smooth(method = "loess", se = FALSE, colour = "#457B9D") +
  scale_y_continuous(labels = scales::comma) +
  scale_x_date(date_labels = "%b %d") +
  labs(
    title = "Daily Yellow Taxi Trips — January 2023",
    x     = "Date",
    y     = "Trip Count"
  ) +
  theme_minimal(base_size = 13)


---

## Performance Tuning

### Creating Indexes

Without indexes, PostgreSQL performs a full table scan for every query — slow for millions of rows. Add indexes on columns used in `WHERE`, `JOIN`, and `ORDER BY` clauses.


In [ ]:
%%R
#| label: create-indexes

# Index on pickup location (used in JOINs)
dbExecute(con, "
  CREATE INDEX IF NOT EXISTS idx_trips_pulocationid
  ON yellow_trips (pulocationid);
")

# Index on dropoff location
dbExecute(con, "
  CREATE INDEX IF NOT EXISTS idx_trips_dolocationid
  ON yellow_trips (dolocationid);
")

# Index on pickup timestamp (for time-range queries)
dbExecute(con, "
  CREATE INDEX IF NOT EXISTS idx_trips_pickup_datetime
  ON yellow_trips (tpep_pickup_datetime);
")

# Composite index for common filter + join pattern
dbExecute(con, "
  CREATE INDEX IF NOT EXISTS idx_trips_location_time
  ON yellow_trips (pulocationid, tpep_pickup_datetime);
")

cat("Indexes created.\n")


### EXPLAIN ANALYZE — Query Performance Diagnostics

`EXPLAIN ANALYZE` shows exactly how PostgreSQL executes a query, including whether indexes are used and how long each step takes.


In [ ]:
%%R
#| label: explain-analyze

explain_result <- dbGetQuery(con, "
  EXPLAIN ANALYZE
  SELECT z.borough, COUNT(*) AS trips
  FROM yellow_trips t
  JOIN taxi_zones z ON t.pulocationid = z.location_id
  GROUP BY z.borough
  ORDER BY trips DESC;
")

cat(explain_result[[1]], sep = "\n")


Look for:
- **Index Scan** (fast) vs **Seq Scan** (slow — no index used)
- **actual time** — milliseconds per node
- **rows** — estimated vs actual row counts

### VACUUM and ANALYZE

After large inserts or deletes, run maintenance:


In [ ]:
%%R
#| label: vacuum

# Update table statistics (helps query planner)
dbExecute(con, "ANALYZE yellow_trips;")

# Reclaim storage from deleted rows (cannot run inside a transaction)
# Run this outside of a transaction block in psql:
# VACUUM ANALYZE yellow_trips;


### Connection Pooling with `pool`

For Shiny apps or multi-user scenarios, use the `pool` package to manage multiple connections efficiently:


In [ ]:
%%R
#| label: connection-pooling

# install.packages("pool")
library(pool)

pool <- dbPool(
  drv      = RPostgres::Postgres(),
  dbname   = Sys.getenv("PG_DB"),
  host     = Sys.getenv("PG_HOST"),
  port     = as.integer(Sys.getenv("PG_PORT")),
  user     = Sys.getenv("PG_USER"),
  password = Sys.getenv("PG_PASSWORD"),
  minSize  = 2,    # Keep at least 2 connections open
  maxSize  = 10    # Allow up to 10 concurrent connections
)

# Use just like a regular connection
result <- dbGetQuery(pool, "SELECT COUNT(*) FROM yellow_trips;")

# Close the pool when done (e.g., when Shiny app shuts down)
poolClose(pool)


---

## Transactions and Error Handling

Transactions group multiple statements into an atomic unit — either all succeed or all are rolled back.


In [ ]:
%%R
#| label: transactions

tryCatch({
  dbBegin(con)

  dbExecute(con, "
    UPDATE employees SET salary = salary * 1.05 WHERE department = 'Engineering';
  ")
  dbExecute(con, "
    INSERT INTO audit_log (event, logged_at)
    VALUES ('Engineering salary increase 5%', NOW());
  ")

  dbCommit(con)
  cat("Transaction committed successfully.\n")

}, error = function(e) {
  dbRollback(con)
  cat("Transaction rolled back due to error:", conditionMessage(e), "\n")
})


### Safe Query Wrapper


In [ ]:
%%R
#| label: safe-query

safe_query <- function(con, sql, params = NULL) {
  tryCatch(
    {
      if (is.null(params)) {
        dbGetQuery(con, sql)
      } else {
        dbGetQuery(con, sql, params = params)
      }
    },
    error = function(e) {
      message("Query failed: ", conditionMessage(e))
      NULL
    }
  )
}

# Usage
result <- safe_query(con, "SELECT * FROM yellow_trips LIMIT 5;")


---

## Database Metadata and Inspection


In [ ]:
%%R
#| label: metadata

# List all tables in the database
dbListTables(con)

# List columns in a table
dbListFields(con, "yellow_trips")

# Detailed column info via SQL
dbGetQuery(con, "
  SELECT
    column_name,
    data_type,
    character_maximum_length,
    is_nullable
  FROM information_schema.columns
  WHERE table_name = 'yellow_trips'
  ORDER BY ordinal_position;
")

# List indexes
dbGetQuery(con, "
  SELECT
    indexname,
    indexdef
  FROM pg_indexes
  WHERE tablename = 'yellow_trips';
")

# Table size on disk
dbGetQuery(con, "
  SELECT
    pg_size_pretty(pg_total_relation_size('yellow_trips')) AS total_size,
    pg_size_pretty(pg_relation_size('yellow_trips'))       AS table_size,
    pg_size_pretty(pg_indexes_size('yellow_trips'))        AS index_size;
")


---

## Writing Query Results Back to PostgreSQL


In [ ]:
%%R
#| label: write-results

# Create a summary in R
borough_summary <- dbGetQuery(con, "
  SELECT
    z.borough,
    COUNT(*)                         AS trip_count,
    ROUND(AVG(t.fare_amount), 2)     AS avg_fare,
    ROUND(SUM(t.total_amount), 2)    AS total_revenue
  FROM yellow_trips t
  JOIN taxi_zones z ON t.pulocationid = z.location_id
  GROUP BY z.borough
  ORDER BY trip_count DESC;
")

# Write summary back as a new table
dbWriteTable(
  conn      = con,
  name      = "borough_summary",
  value     = borough_summary,
  overwrite = TRUE,
  row.names = FALSE
)

# Verify
dbGetQuery(con, "SELECT * FROM borough_summary;")


---

## Security Best Practices

| Practice | Why It Matters |
|----------|----------------|
| Never hardcode passwords in scripts | Scripts are often committed to Git |
| Use `.Renviron` or `keyring` | Keeps credentials out of source code |
| Create a dedicated database user | Avoid using the `postgres` superuser |
| Grant minimum required privileges | Principle of least privilege |
| Use parameterised queries | Prevents SQL injection |
| Encrypt connections with SSL | Protects data in transit |
| Rotate passwords regularly | Reduces risk from credential leaks |

### Enabling SSL Connections


In [ ]:
%%R
#| label: ssl-connection

con_ssl <- dbConnect(
  drv      = RPostgres::Postgres(),
  dbname   = Sys.getenv("PG_DB"),
  host     = Sys.getenv("PG_HOST"),
  port     = 5432,
  user     = Sys.getenv("PG_USER"),
  password = Sys.getenv("PG_PASSWORD"),
  sslmode  = "require"   # Options: disable, allow, prefer, require, verify-ca, verify-full
)


### Revoking Unnecessary Privileges

```sql
-- In psql as postgres superuser
REVOKE ALL ON DATABASE taxi_db FROM PUBLIC;
GRANT CONNECT ON DATABASE taxi_db TO r_user;
GRANT USAGE ON SCHEMA public TO r_user;
GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA public TO r_user;
```

---

## Cleanup and Disconnecting


In [ ]:
%%R
#| label: cleanup

# Drop working tables (in development / tutorial contexts)
dbExecute(con, "DROP TABLE IF EXISTS yellow_trips;")
dbExecute(con, "DROP TABLE IF EXISTS taxi_zones;")
dbExecute(con, "DROP TABLE IF EXISTS borough_summary;")
dbExecute(con, "DROP TABLE IF EXISTS employees;")

# Always disconnect when done — releases the server connection
dbDisconnect(con)
cat("Disconnected from PostgreSQL.\n")


---

## Quick Reference Cheat Sheet

### DBI Functions

| Function | Purpose |
|----------|---------|
| `dbConnect()` | Open a database connection |
| `dbDisconnect()` | Close the connection |
| `dbGetQuery()` | Run a SELECT query, return data frame |
| `dbExecute()` | Run INSERT/UPDATE/DELETE/DDL, return row count |
| `dbWriteTable()` | Write an R data frame to a table |
| `dbReadTable()` | Read a whole table into R |
| `dbListTables()` | List all tables in the database |
| `dbListFields()` | List columns in a table |
| `dbExistsTable()` | Check if a table exists |
| `dbBegin()` | Start a transaction |
| `dbCommit()` | Commit a transaction |
| `dbRollback()` | Roll back a transaction |
| `dbSendQuery()` | Send query (for streaming / batched fetch) |
| `dbFetch()` | Fetch rows from a sent query |
| `dbClearResult()` | Release resources from a sent query |

### Common PostgreSQL SQL Patterns

```sql
-- Count rows
SELECT COUNT(*) FROM table_name;

-- Distinct values
SELECT DISTINCT column FROM table_name;

-- Top N rows
SELECT * FROM table_name ORDER BY column DESC LIMIT 10;

-- Conditional aggregation
SELECT
  SUM(CASE WHEN condition THEN value ELSE 0 END) AS conditional_sum
FROM table_name;

-- Date truncation
SELECT DATE_TRUNC('hour', timestamp_col) AS hour, COUNT(*)
FROM table_name GROUP BY 1;

-- String formatting
SELECT TO_CHAR(amount, 'FM$999,999.00') AS formatted_amount FROM table_name;

-- Window function: running total
SELECT date, amount,
  SUM(amount) OVER (ORDER BY date) AS running_total
FROM table_name;
```

---

## Summary

This tutorial covered the full workflow for using PostgreSQL with R:

1. **Installation** — Setting up PostgreSQL on Ubuntu and Windows
2. **Database creation** — Using `psql` and pgAdmin
3. **R connection** — `DBI` + `RPostgres`, secure credential management
4. **CRUD operations** — Create, Read, Update, Delete with `dbExecute()` and `dbGetQuery()`
5. **Data import** — CSV with `readr`, Parquet with `arrow`, chunked insertion for large files
6. **Advanced SQL** — JOINs, CTEs, window functions, time-series queries
7. **dbplyr** — Writing dplyr code that runs as SQL on the server
8. **Performance** — Indexes, `EXPLAIN ANALYZE`, connection pooling
9. **Transactions** — Atomic multi-statement operations with error handling
10. **Security** — `.Renviron`, parameterised queries, SSL, least-privilege users

### Next Steps

- Explore **[dbplyr](https://dbplyr.tidyverse.org/)** for advanced SQL translation
- Use **[pool](https://rstudio.github.io/pool/)** for production Shiny apps
- Consider **[TimescaleDB](https://www.timescale.com/)** for time-series extensions
- Learn **PostgreSQL COPY** command for ultra-fast bulk imports
- Investigate **PostGIS** for geospatial data (mapping taxi zones, route analysis)

---

> **Tip: Further Reading**
>
- [DBI Package Documentation](https://dbi.r-dbi.org/)
- [RPostgres GitHub](https://github.com/r-dbi/RPostgres)
- [dbplyr Documentation](https://dbplyr.tidyverse.org/)
- [PostgreSQL Official Documentation](https://www.postgresql.org/docs/)
- [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
